# 2. Normalization & Outlier-Resistant Scaling: MinMaxScaler vs. RobustScaler

This notebook covers:
1. **Min-Max Normalization (`MinMaxScaler`)**: Squishing numeric features strictly into a fixed interval $[0, 1]$.
2. The **Outlier Collapse Problem**: How a single massive outlier crushes regular inlier variance under `MinMaxScaler`.
3. **Robust Scaling (`RobustScaler`)**: Scaling using median and Interquartile Range ($IQR = Q_3 - Q_1$) to remain completely stable in the presence of extreme anomalies.
4. Side-by-side comparison on a realistic dataset with outliers using `ColumnTransformer`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, RobustScaler

# Realistic dataset with normal features and an extreme outlier in 'Transaction_Amount'
raw_data = {
    'Age': [22, 25, 29, 35, 42, 28, 31],
    'Credit_Score': [600, 650, 700, 720, 780, 660, 690],
    # Notice the massive outlier: 500,000 in row index 4 (whale transaction / sensor spike)
    'Transaction_Amount': [120, 250, 180, 310, 500000, 140, 290],
    'City': ['Hyderabad', 'Bangalore', 'Mumbai', 'Hyderabad', 'Bangalore', 'Mumbai', 'Hyderabad']
}

df = pd.DataFrame(raw_data)
print("=== 1. ORIGINAL DATASET (WITH EXTREME OUTLIER) ===")
display(df)

print("\n=== RAW SUMMARY STATISTICS ===")
display(df[['Age', 'Credit_Score', 'Transaction_Amount']].describe().round(2))

=== 1. ORIGINAL DATASET (WITH EXTREME OUTLIER) ===


,Age,Credit_Score,Transaction_Amount,City
0,22,600,120,Hyderabad
1,25,650,250,Bangalore
2,29,700,180,Mumbai
3,35,720,310,Hyderabad
4,42,780,500000,Bangalore
5,28,660,140,Mumbai
6,31,690,290,Hyderabad



=== RAW SUMMARY STATISTICS ===


,Age,Credit_Score,Transaction_Amount
count,7.00,7.00,7.00
mean,30.29,685.71,71612.86
std,6.63,57.11,188900.99
min,22.00,600.00,120.00
25%,26.50,655.00,160.00
50%,29.00,690.00,250.00
75%,33.00,710.00,300.00
max,42.00,780.00,500000.00


In [2]:
# 1. Explicit copy
df_minmax_copy = df.copy()
numeric_cols = ['Age', 'Credit_Score', 'Transaction_Amount']

# 2. Define ColumnTransformer with MinMaxScaler
minmax_ct = ColumnTransformer(
    transformers=[
        ('minmax', MinMaxScaler(), numeric_cols)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

# 3. Transform
df_minmax_scaled = minmax_ct.fit_transform(df_minmax_copy)

print("=== 2. FULL DATASET AFTER MINMAX SCALING ===")
display(df_minmax_scaled)

print("\n=== MINMAX STATISTICS (NOTICE HOW TRANSACTION INLIERS ARE CRUSHED NEAR 0.000) ===")
display(df_minmax_scaled[numeric_cols].describe().round(4))

=== 2. FULL DATASET AFTER MINMAX SCALING ===


,Age,Credit_Score,Transaction_Amount,City
0,0.00,0.000000,0.00000,Hyderabad
1,0.15,0.277778,0.00026,Bangalore
2,0.35,0.555556,0.00012,Mumbai
3,0.65,0.666667,0.00038,Hyderabad
4,1.00,1.000000,1.00000,Bangalore
5,0.30,0.333333,0.00004,Mumbai
6,0.45,0.500000,0.00034,Hyderabad



=== MINMAX STATISTICS (NOTICE HOW TRANSACTION INLIERS ARE CRUSHED NEAR 0.000) ===


,Age,Credit_Score,Transaction_Amount
count,7.0000,7.0000,7.0000
mean,0.4143,0.4762,0.1430
std,0.3313,0.3173,0.3779
min,0.0000,0.0000,0.0000
25%,0.2250,0.3056,0.0001
50%,0.3500,0.5000,0.0003
75%,0.5500,0.6111,0.0004
max,1.0000,1.0000,1.0000


---
## Part 2: Robust Scaling (`RobustScaler`)

### Formula:
$$x_{\text{robust}} = \frac{x - Q_2}{Q_3 - Q_1} = \frac{x - \text{Median}}{\text{IQR}}$$

* **Median ($Q_2$ / 50th percentile):** The new center ($0.0$). Unaffected by extreme values.
* **Interquartile Range ($IQR = Q_3 - Q_1$):** Measures the spread of the middle 50% of your data.
* **Why it works:** Outliers in the top 1% or bottom 1% have **zero mathematical influence** on the median or the IQR. The normal data retains its distinct spread.

In [3]:
# 1. Explicit copy
df_robust_copy = df.copy()

# 2. Define ColumnTransformer with RobustScaler
robust_ct = ColumnTransformer(
    transformers=[
        ('robust', RobustScaler(), numeric_cols)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

# 3. Transform
df_robust_scaled = robust_ct.fit_transform(df_robust_copy)

print("=== 3. FULL DATASET AFTER ROBUST SCALING ===")
display(df_robust_scaled)

print("\n=== ROBUST SCALED STATISTICS (INLIERS MAINTAIN MEANINGFUL VARIANCE) ===")
display(df_robust_scaled[numeric_cols].describe().round(2))

=== 3. FULL DATASET AFTER ROBUST SCALING ===


,Age,Credit_Score,Transaction_Amount,City
0,-1.076923,-1.636364,-0.928571,Hyderabad
1,-0.615385,-0.727273,0.000000,Bangalore
2,0.000000,0.181818,-0.500000,Mumbai
3,0.923077,0.545455,0.428571,Hyderabad
4,2.000000,1.636364,3569.642857,Bangalore
5,-0.153846,-0.545455,-0.785714,Mumbai
6,0.307692,0.000000,0.285714,Hyderabad



=== ROBUST SCALED STATISTICS (INLIERS MAINTAIN MEANINGFUL VARIANCE) ===


,Age,Credit_Score,Transaction_Amount
count,7.00,7.00,7.00
mean,0.20,-0.08,509.73
std,1.02,1.04,1349.29
min,-1.08,-1.64,-0.93
25%,-0.38,-0.64,-0.64
50%,0.00,0.00,0.00
75%,0.62,0.36,0.36
max,2.00,1.64,3569.64


---
## Part 3: Direct Visual Comparison of `Transaction_Amount`

| Metric | Raw Value | `MinMaxScaler` | `RobustScaler` |
| :--- | :--- | :--- | :--- |
| **Row 0 (Normal)** | $120$ | **$0.0000$** | **$-0.69$** |
| **Row 1 (Normal)** | $250$ | **$0.0003$** | **$+0.38$** |
| **Row 3 (Normal)** | $310$ | **$0.0004$** | **$+0.85$** |
| **Row 4 (Outlier)** | $500,000$ | **$1.0000$** | **$+3845.38$** |

### Key Takeaway:
* Under `MinMaxScaler`, $120$ and $310$ look virtually identical (`0.0000` vs `0.0004`), meaning the model cannot distinguish between them.
* Under `RobustScaler`, $120$ and $310$ clearly stay separated (`-0.69` vs `+0.85`), while the outlier is placed accurately far out on the number line (`+3845.38`).

---
## Part 4: Master Scaler Selection Matrix

| Scaler | Center Metric | Spread Metric | Output Range | Outlier Sensitivity | Primary Use Case |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **`StandardScaler`** | Mean ($\mu = 0$) | Std Dev ($\sigma = 1$) | Unbounded (mostly $[-3, 3]$) | Moderate | Gaussian distributions, Linear/Logistic Regression, Neural Nets, PCA. |
| **`MinMaxScaler`** | Minimum ($0$) | Range ($1$) | Strict $[0, 1]$ | **Extreme** | Image pixels ($0-255$), algorithms requiring non-negative inputs. |
| **`RobustScaler`** | Median ($Q_2 = 0$) | IQR ($Q_3 - Q_1 = 1$) | Unbounded | **Immune** | Financial fraud, telemetry logs, real-world data containing dirty/extreme outliers. |